In [1]:
!pip -q install pandas numpy requests tqdm

In [2]:
from google.colab import files
import pandas as pd
import numpy as np
from pathlib import Path

uploaded = files.upload()
input_file = next(iter(uploaded))

df = pd.read_csv(input_file, low_memory=False)

df["sequence"] = (
    df["sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Input:", input_file)
print("Rows:", len(df))
print("Unique:", df["sequence"].nunique())
print("Length range:", df["sequence"].str.len().min(), "-", df["sequence"].str.len().max())

assert len(df) == 60000
assert df["sequence"].nunique() == 60000

Saving STEP9B_60K_WITH_BIOLOGICAL_EMBEDDING_SCORES.csv to STEP9B_60K_WITH_BIOLOGICAL_EMBEDDING_SCORES.csv
Input: STEP9B_60K_WITH_BIOLOGICAL_EMBEDDING_SCORES.csv
Rows: 60000
Unique: 60000
Length range: 9 - 25


In [3]:
ALLOWED = set("ACDEFGHIKLMNPQRSTVWY")

invalid = df["sequence"].map(
    lambda s: (
        len(s) < 8
        or len(s) > 50
        or not set(s).issubset(ALLOWED)
    )
)

print("Invalid sequences:", int(invalid.sum()))

if invalid.any():
    display(df.loc[invalid, ["sequence"]].head())

assert invalid.sum() == 0

Invalid sequences: 0


In [4]:
df["peptide_length"] = df["sequence"].str.len()

df["pepsysco_in_validated_length_domain"] = (
    df["peptide_length"].between(8, 25)
)

print(
    df["pepsysco_in_validated_length_domain"]
    .value_counts()
)

print(
    "\n8-25 aa:",
    int(df["pepsysco_in_validated_length_domain"].sum())
)

print(
    ">25 aa:",
    int((df["peptide_length"] > 25).sum())
)

pepsysco_in_validated_length_domain
True    60000
Name: count, dtype: int64

8-25 aa: 60000
>25 aa: 0


In [5]:
ALIPHATIC_HYDROPHOBIC = set("ILMV")
AROMATIC = set("FWY")
ACIDIC = set("DE")
BASIC = set("HKR")
SMALL_POLAR = set("CST")
SMALL = set("AGP")
LARGE_POLAR = set("NQ")

def longest_stretch(seq, residue_set):
    best = 0
    current = 0

    for aa in seq:
        if aa in residue_set:
            current += 1
            best = max(best, current)
        else:
            current = 0

    return best

df["synth_aliphatic_hydrophobic_count_ILMV"] = (
    df["sequence"].map(
        lambda s: sum(aa in ALIPHATIC_HYDROPHOBIC for aa in s)
    )
)

df["synth_longest_ILMV_stretch"] = (
    df["sequence"].map(
        lambda s: longest_stretch(
            s,
            ALIPHATIC_HYDROPHOBIC
        )
    )
)

df["synth_n_terminal_residue"] = (
    df["sequence"].str[0]
)

df["synth_acidic_count_DE"] = (
    df["sequence"].map(
        lambda s: sum(aa in ACIDIC for aa in s)
    )
)

df["synth_basic_count_HKR"] = (
    df["sequence"].map(
        lambda s: sum(aa in BASIC for aa in s)
    )
)

df["synth_aromatic_count_FWY"] = (
    df["sequence"].map(
        lambda s: sum(aa in AROMATIC for aa in s)
    )
)

df["synth_small_polar_count_CST"] = (
    df["sequence"].map(
        lambda s: sum(aa in SMALL_POLAR for aa in s)
    )
)

df["synth_small_count_AGP"] = (
    df["sequence"].map(
        lambda s: sum(aa in SMALL for aa in s)
    )
)

df["synth_large_polar_count_NQ"] = (
    df["sequence"].map(
        lambda s: sum(aa in LARGE_POLAR for aa in s)
    )
)

# Individual residues useful for audit/diagnostic only
df["synth_cys_count"] = df["sequence"].str.count("C")
df["synth_met_count"] = df["sequence"].str.count("M")
df["synth_pro_count"] = df["sequence"].str.count("P")
df["synth_trp_count"] = df["sequence"].str.count("W")

print("Diagnostics calculated.")

Diagnostics calculated.


In [6]:
valid_subset = df.loc[
    df["pepsysco_in_validated_length_domain"],
    ["sequence"]
].copy()

valid_subset.to_csv(
    "/content/STEP10_PEPSYSCO_INPUT_8_25.txt",
    index=False,
    header=False
)

print("Sequences for PepSySco:", len(valid_subset))
print("Saved: /content/STEP10_PEPSYSCO_INPUT_8_25.txt")

Sequences for PepSySco: 60000
Saved: /content/STEP10_PEPSYSCO_INPUT_8_25.txt


In [7]:
from google.colab import files

files.download(
    "/content/STEP10_PEPSYSCO_INPUT_8_25.txt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from google.colab import files

uploaded = files.upload()
pepsysco_file = next(iter(uploaded))

pep = pd.read_csv(pepsysco_file)

print("PepSySco rows:", len(pep))
print("Columns:", list(pep.columns))

display(pep.head())

Saving result.csv to result.csv
PepSySco rows: 60000
Columns: ['peptide', 'score']


,peptide,score
0,GKWFVKLKC,0.99998
1,GLKLCYYRW,0.99995
2,FKRLLHKFRC,0.99986
3,FRLWKKLKFF,0.99986
4,IKKRIFMWRW,0.99980


In [9]:
seq_candidates = [
    c for c in pep.columns
    if "peptide" in c.lower()
    or "sequence" in c.lower()
]

score_candidates = [
    c for c in pep.columns
    if "score" in c.lower()
]

print("Possible sequence columns:", seq_candidates)
print("Possible score columns:", score_candidates)

if len(seq_candidates) != 1:
    raise ValueError(
        "Could not uniquely identify peptide column."
    )

if len(score_candidates) != 1:
    raise ValueError(
        "Could not uniquely identify PepSySco score column."
    )

pep_seq_col = seq_candidates[0]
pep_score_col = score_candidates[0]

print("Sequence column:", pep_seq_col)
print("Score column:", pep_score_col)

Possible sequence columns: ['peptide']
Possible score columns: ['score']
Sequence column: peptide
Score column: score


In [10]:
pep = pep[
    [pep_seq_col, pep_score_col]
].copy()

pep.columns = [
    "sequence",
    "pepsysco_score"
]

pep["sequence"] = (
    pep["sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

pep["pepsysco_score"] = pd.to_numeric(
    pep["pepsysco_score"],
    errors="coerce"
)

print("PepSySco rows:", len(pep))
print("Unique:", pep["sequence"].nunique())
print("Missing score:", pep["pepsysco_score"].isna().sum())

assert pep["sequence"].duplicated().sum() == 0

df = df.merge(
    pep,
    on="sequence",
    how="left",
    validate="one_to_one"
)

PepSySco rows: 60000
Unique: 60000
Missing score: 0


In [11]:
df["pepsysco_ge_0_85"] = (
    df["pepsysco_score"] >= 0.85
)

df["pepsysco_ge_0_99"] = (
    df["pepsysco_score"] >= 0.99
)

# Outside the validated 8–25 aa subset, keep score missing
# rather than extrapolating or inventing a surrogate score.

outside = ~df["pepsysco_in_validated_length_domain"]

df.loc[
    outside,
    [
        "pepsysco_score",
        "pepsysco_ge_0_85",
        "pepsysco_ge_0_99"
    ]
] = np.nan

/tmp/ipykernel_557/1056622788.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[
/tmp/ipykernel_557/1056622788.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[


In [12]:
inside = df["pepsysco_in_validated_length_domain"]

print("=" * 70)
print("STEP 10 VALIDATION")
print("=" * 70)

print("Total rows:", len(df))
print("Unique sequences:", df["sequence"].nunique())

print(
    "Inside PepSySco validated domain:",
    int(inside.sum())
)

print(
    "Outside validated domain:",
    int((~inside).sum())
)

print(
    "PepSySco scores present inside domain:",
    int(df.loc[inside, "pepsysco_score"].notna().sum())
)

print(
    "PepSySco scores missing inside domain:",
    int(df.loc[inside, "pepsysco_score"].isna().sum())
)

print("\nPepSySco score summary:")
print(
    df.loc[
        inside,
        "pepsysco_score"
    ].describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
    )
)

print(
    "\nPepSySco >= 0.85:",
    int(
        (
            df.loc[inside, "pepsysco_score"]
            >= 0.85
        ).sum()
    )
)

print(
    "PepSySco >= 0.99:",
    int(
        (
            df.loc[inside, "pepsysco_score"]
            >= 0.99
        ).sum()
    )
)

print("\nLongest ILMV stretch:")
print(
    df["synth_longest_ILMV_stretch"]
    .describe()
)

print("\nCys count:")
print(
    df["synth_cys_count"]
    .describe()
)

STEP 10 VALIDATION
Total rows: 60000
Unique sequences: 60000
Inside PepSySco validated domain: 60000
Outside validated domain: 0
PepSySco scores present inside domain: 60000
PepSySco scores missing inside domain: 0

PepSySco score summary:
count    60000.000000
mean         0.841128
std          0.100052
min          0.295350
5%           0.645950
25%          0.783520
50%          0.863330
75%          0.918650
95%          0.963790
max          0.999980
Name: pepsysco_score, dtype: float64

PepSySco >= 0.85: 32902
PepSySco >= 0.99: 233

Longest ILMV stretch:
count    60000.000000
mean         2.351433
std          1.416037
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         14.000000
Name: synth_longest_ILMV_stretch, dtype: float64

Cys count:
count    60000.000000
mean         0.520783
std          1.016408
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         11.000000
Name: synth_cys_coun

In [13]:
assert len(df) == 60000
assert df["sequence"].nunique() == 60000

assert (
    df.loc[
        inside,
        "pepsysco_score"
    ]
    .dropna()
    .between(0, 1)
    .all()
)

assert (
    df.loc[
        ~inside,
        "pepsysco_score"
    ]
    .isna()
    .all()
)

print("All Step 10 validation checks passed.")

All Step 10 validation checks passed.


In [14]:
OUT = (
    "/content/"
    "STEP10_60K_WITH_PEPSYSCO_SYNTHESIZABILITY.csv"
)

df.to_csv(
    OUT,
    index=False
)

print("Saved:", OUT)

Saved: /content/STEP10_60K_WITH_PEPSYSCO_SYNTHESIZABILITY.csv


In [15]:
import json

manifest = {
    "step": "10",
    "method": "PepSySco + literature-defined synthesis diagnostics",
    "primary_predictor": "PepSySco",
    "pepsysco_model": "Gaussian Naive Bayes",
    "pepsysco_final_features": [
        "peptide_length",
        "Janin_hydrophobicity_index"
    ],
    "validated_length_domain": "8-25 aa",
    "published_reference_thresholds": {
        "0.85": {
            "reported_coverage": "~51%",
            "reported_accuracy": "~95%"
        },
        "0.99": {
            "reported_coverage": "~12%",
            "reported_accuracy": "~98%"
        }
    },
    "additional_diagnostics": [
        "ILMV_count",
        "longest_ILMV_stretch",
        "N_terminal_residue",
        "DE_count",
        "HKR_count",
        "FWY_count",
        "CST_count",
        "AGP_count",
        "NQ_count",
        "Cys_count",
        "Met_count",
        "Pro_count",
        "Trp_count"
    ],
    "hard_filter_applied": False,
    "rows": int(len(df)),
    "unique_sequences": int(
        df["sequence"].nunique()
    )
}

with open(
    "/content/STEP10_manifest.json",
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

print("Saved: /content/STEP10_manifest.json")

Saved: /content/STEP10_manifest.json


In [16]:
from google.colab import files

files.download(
    "/content/"
    "STEP10_60K_WITH_PEPSYSCO_SYNTHESIZABILITY.csv"
)

files.download(
    "/content/STEP10_manifest.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>